In [ ]:
%pip install "pyspark==3.5.1" "delta-spark==3.1.0"

# Create local lakehouse

In this notebook we will create a lakehouse locally, and then do some cool things.

In [2]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

# Stop any existing session first (if running inside a notebook)
try:
    spark.stop()
except:
    pass

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("delta-pip-test")
    # Delta
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # Local dirs (Windows-friendly)
    .config("spark.sql.warehouse.dir", r"C:\dev\github\FabCon\lakehouse")
    .config("spark.local.dir", r"C:\dev\github\FabCon\spark_tmp")
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "127.0.0.1")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:

# Bronze: ingest CSV
df = spark.read.parquet(r"C:\dev\github\FabCon\lakehouse\Files\green_tripdata_2022-08.parquet")


from pathlib import Path

table_name = "green_tripdata_2022_08"
base = Path(r"C:\dev\github\FabCon\lakehouse\Tables")
base.mkdir(parents=True, exist_ok=True)
table_path = (base / table_name).as_posix()

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .save(table_path)
)

spark.sql(f"DROP TABLE IF EXISTS {table_name}")
spark.sql(f"CREATE TABLE {table_name} USING DELTA LOCATION '{table_path}'")

spark.table(table_name).show(5, truncate=False)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|2       |2022-08-01 00:18:31 |2022-08-01 00:31:33  |N                 |1.0       |80          |225         |1.0            |2.9          |11.5       |0.5  |0.5   

# Scenario 1: Data cleaning and transformation

In this scenario, the data engineer could perform some data cleaning and transformation tasks to prepare the data for downstream analysis. For example, they could remove any invalid or missing data, convert the data types of some columns, and add some new calculated columns based on the existing data. This could involve using Spark SQL queries, user-defined functions (UDFs), or built-in Spark functions to manipulate the data.

In [5]:
from pyspark.sql.functions import col, when

# Load data from source
df = spark.table("green_tripdata_2022_08")
df_count = df.count()

print(f"Total records before cleansing: {df_count}")

# Remove invalid records
df = df.filter((col("trip_distance") > 0) & (col("fare_amount") > 0))
df_count_after_clearning = df.count()

number_of_deleted_records = df_count - df_count_after_clearning

print(f"Removed {number_of_deleted_records} records")

# # Cleanse data
df = df.withColumn("store_and_fwd_flag", when(col("store_and_fwd_flag") == "Y", True).otherwise(False))
df = df.withColumn("lpep_pickup_datetime", col("lpep_pickup_datetime").cast("timestamp"))
df = df.withColumn("lpep_dropoff_datetime", col("lpep_dropoff_datetime").cast("timestamp"))

# Display cleansed data to destination
display(df)

# Write cleansed data to destination
table_name = "green_tripdata_2022_08_cleansed"
table_path = (base / table_name).as_posix()

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .save(table_path)
)

spark.sql(f"DROP TABLE IF EXISTS {table_name}")
spark.sql(f"CREATE TABLE {table_name} USING DELTA LOCATION '{table_path}'")


Total records before cleansing: 65929
Removed 4761 records


DataFrame[VendorID: bigint, lpep_pickup_datetime: timestamp, lpep_dropoff_datetime: timestamp, store_and_fwd_flag: boolean, RatecodeID: double, PULocationID: bigint, DOLocationID: bigint, passenger_count: double, trip_distance: double, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, ehail_fee: int, improvement_surcharge: double, total_amount: double, payment_type: double, trip_type: double, congestion_surcharge: double]

DataFrame[]

In [6]:
df.show(10)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2022-08-01 00:18:31|  2022-08-01 00:31:33|             false|       1.0|          80|         225|            1.0|          2.9|       11.5|  0.5|    0.